In [65]:
# language modeling is the task of prediciting next character or word in a sequence based on the context of previous words. 

# This is Unsupervised DataSet 
# -> Python library for DataScience

#  We convert this into Supervised DataSet  
#    Input                   Output 
#    Python                  -> library
#    Python , library        -> for 
#    Python ,library , for   -> DataScience


# Convert the words into numbers as NN/Maths don't work with words it works with Numbers
# Assign unique words to the each word
# Python - 1 , library - 2 , for - 3 , DataScience - 4

# It can be sent to the model but this method does not capture the meaning of the words and combine them model might think that dog - 5 , cat - 4 , mouse - 6 , are related to each other , therefor we generate Embeddings of this words which brings words with similar meanings closer to each other
  
#  Embeddings are made with NN. 

In [1]:
import torch 
from torch.utils.data import Dataset,DataLoader 
import torch.nn as nn
import torch.optim as optim

In [2]:
from collections import Counter

In [ ]:
# Sample Data of FAQ from a Website
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.link
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the  for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""


In [4]:
import re

def tokenizer(text: str):
    """
    Tokenizes the given text into words, numbers, and special characters.
    Similar to nltk.word_tokenize but lightweight.
    """
    # Regex explanation:
    #   \w+    -> words and numbers
    #   [^\w\s] -> any symbol (punctuation, emoji, etc.)
    #   \s+ ignored automatically
    # We told regex: “prefer long runs of word characters (\w+) — those are words/numbers/underscore — otherwise take single non-word non-space characters.”
    tokens = re.findall(r"\w+|[^\w\s]", text, re.UNICODE)
    return tokens

In [5]:
import re

TOKEN_RE = re.compile(r"""
    (https?://[^\s]+)                       # URLs
  | ([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})  # emails
  | ([#@]\w+)                              # hashtags or mentions
  | (\d+(?:\.\d+)?)                        # numbers with optional decimal
  | (\w+(?:['’-]\w+)*)                     # words with internal apostrophes/hyphens
  | (\.{2,})                               # ellipsis or multi-dot
  | ([^\w\s])                              # any other single non-word, non-space char
  """, re.VERBOSE | re.UNICODE)

def improved_tokenize(text: str):
    return [m.group(0) for m in TOKEN_RE.finditer(text)]


In [6]:
tokens = improved_tokenize(document.lower())

In [7]:
Counter(tokens).keys

<function Counter.keys>

In [8]:
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<unk>': 0,
 'about': 1,
 'the': 2,
 'program': 3,
 'what': 4,
 'is': 5,
 'course': 6,
 'fee': 7,
 'for': 8,
 'data': 9,
 'science': 10,
 'mentorship': 11,
 '(': 12,
 'dsmp': 13,
 '2023': 14,
 ')': 15,
 'follows': 16,
 'a': 17,
 'monthly': 18,
 'subscription': 19,
 'model': 20,
 'where': 21,
 'you': 22,
 'have': 23,
 'to': 24,
 'make': 25,
 'payments': 26,
 'of': 27,
 'rs': 28,
 '799': 29,
 '/': 30,
 'month': 31,
 '.': 32,
 'total': 33,
 'duration': 34,
 '?': 35,
 '7': 36,
 'months': 37,
 'so': 38,
 'becomes': 39,
 '*': 40,
 '=': 41,
 '5600': 42,
 'approx': 43,
 'syllabus': 44,
 'we': 45,
 'will': 46,
 'be': 47,
 'covering': 48,
 'following': 49,
 'modules': 50,
 ':': 51,
 'python': 52,
 'fundamentals': 53,
 'libraries': 54,
 'analysis': 55,
 'sql': 56,
 'maths': 57,
 'machine': 58,
 'learning': 59,
 'ml': 60,
 'algorithms': 61,
 'practical': 62,
 'mlops': 63,
 'case': 64,
 'studies': 65,
 'can': 66,
 'check': 67,
 'detailed': 68,
 'here': 69,
 '-': 70,
 'https://learnwith.campusx.in/

In [9]:
len(vocab)

281

In [10]:
vocab['about']

1

In [11]:
input_sentences = document.split('\n')

In [12]:
input_sentences

['About the Program',
 'What is the course fee for  Data Science Mentorship Program (DSMP 2023)',
 'The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.',
 'What is the total duration of the course?',
 'The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)',
 'What is the syllabus of the mentorship program?',
 'We will be covering the following modules:',
 'Python Fundamentals',
 'Python libraries for Data Science',
 'Data Analysis',
 'SQL for Data Science',
 'Maths for Machine Learning',
 'ML Algorithms',
 'Practical ML',
 'MLOPs',
 'Case studies',
 'You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390',
 'Will Deep Learning and NLP be a part of this program?',
 'No, NLP and Deep Learning both are not a part of this program’s curriculum.',
 'What if I miss a live session? Will I get a recording 

In [13]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence

In [14]:
input_numerical_sentences = [] 

In [15]:
for sentence in input_sentences:
    print(improved_tokenize(sentence.lower()))
    print(text_to_indices(improved_tokenize(sentence.lower()),vocab))
    input_numerical_sentences.append(text_to_indices(improved_tokenize(sentence.lower()),vocab))

['about', 'the', 'program']
[1, 2, 3]
['what', 'is', 'the', 'course', 'fee', 'for', 'data', 'science', 'mentorship', 'program', '(', 'dsmp', '2023', ')']
[4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14, 15]
['the', 'course', 'follows', 'a', 'monthly', 'subscription', 'model', 'where', 'you', 'have', 'to', 'make', 'monthly', 'payments', 'of', 'rs', '799', '/', 'month', '.']
[2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27, 28, 29, 30, 31, 32]
['what', 'is', 'the', 'total', 'duration', 'of', 'the', 'course', '?']
[4, 5, 2, 33, 34, 27, 2, 6, 35]
['the', 'total', 'duration', 'of', 'the', 'course', 'is', '7', 'months', '.', 'so', 'the', 'total', 'course', 'fee', 'becomes', '799', '*', '7', '=', 'rs', '5600', '(', 'approx', '.', ')']
[2, 33, 34, 27, 2, 6, 5, 36, 37, 32, 38, 2, 33, 6, 7, 39, 29, 40, 36, 41, 28, 42, 12, 43, 32, 15]
['what', 'is', 'the', 'syllabus', 'of', 'the', 'mentorship', 'program', '?']
[4, 5, 2, 44, 27, 2, 11, 3, 35]
['we', 'will', 'be', 'covering', 'the', 'following

In [16]:
len(input_numerical_sentences)

78

In [17]:
training_sequence = []
for sentence in input_numerical_sentences:

    for i in range(1,len(sentence)):
        training_sequence.append(sentence[:i+1])

In [18]:
training_sequence

[[1, 2],
 [1, 2, 3],
 [4, 5],
 [4, 5, 2],
 [4, 5, 2, 6],
 [4, 5, 2, 6, 7],
 [4, 5, 2, 6, 7, 8],
 [4, 5, 2, 6, 7, 8, 9],
 [4, 5, 2, 6, 7, 8, 9, 10],
 [4, 5, 2, 6, 7, 8, 9, 10, 11],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14, 15],
 [2, 6],
 [2, 6, 16],
 [2, 6, 16, 17],
 [2, 6, 16, 17, 18],
 [2, 6, 16, 17, 18, 19],
 [2, 6, 16, 17, 18, 19, 20],
 [2, 6, 16, 17, 18, 19, 20, 21],
 [2, 6, 16, 17, 18, 19, 20, 21, 22],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27, 28],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18

In [19]:
# Number of Training Rows 
len(training_sequence)

935

In [85]:
# give 32 sentences (batch) as input, parallely(only if input size is same therefor we add padding into the data) LSTM does forward propagation , loss of the entire batch is calculated , according to that loss backpropagation is made, and Weights gets updated. 


# for that we need the largest row in the data

In [20]:
len_list = []
for sequence in training_sequence:
    len_list.append(len(sequence)) 

max_length = max(len_list)

In [21]:
padded_training_sequence = []
for sequence in training_sequence:
    padded_training_sequence.append([0]* (max(len_list) - len(sequence))+sequence)

In [22]:
print(len(padded_training_sequence))
print(padded_training_sequence[5])

935
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 5, 2, 6, 7]


In [23]:
padded_training_sequence = torch.tensor(padded_training_sequence,dtype=torch.long)

In [24]:
padded_training_sequence.shape

torch.Size([935, 67])

In [25]:
X = padded_training_sequence[:,:-1]
y = padded_training_sequence[:,-1]

In [26]:
class CustomDataset(Dataset):
    def __init__(self , X ,y ):
        self.X = X 
        self.y = y

    def __len__(self):
        return self.X.shape[0]
        
    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]

In [27]:
dataset = CustomDataset(X,y)

In [28]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 tensor(2))

In [29]:
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)

In [30]:
dataloader

In [31]:
len_of_batches = 0 
for input, output in dataloader:
    len_of_batches = len_of_batches + 1
    print(input , output)

print("Total batch:" , len_of_batches)

tensor([[  0,   0,   0,  ...,   4, 202,  85],
        [  0,   0,   0,  ...,   0,   0, 136],
        [  0,   0,   0,  ..., 139, 148, 149],
        ...,
        [  0,   0,   0,  ..., 138,   2,   3],
        [  0,   0,   0,  ...,  23,  24, 150],
        [  0,   0,   0,  ..., 224, 225, 176]]) tensor([132,  78,  73,  37,  81, 136, 144,   2,  85, 151,  47,  23,  24, 133,
        254, 106, 139,  46,  32,  36, 143,  89,   2,  66,   2,   2,  99,  32,
        142, 141,   2,   5])
tensor([[  0,   0,   0,  ...,  85,  23,  24],
        [  0,   0,   0,  ..., 117, 118,   2],
        [  0,   0,   0,  ...,  78,  85, 133],
        ...,
        [  0,   0,   0,  ..., 127, 128,  88],
        [  0,   0,   0,  ..., 189,  22,  23],
        [  0,   0,   0,  ...,   0, 175,  77]]) tensor([150,  94, 269,  24,  78,  85,  37, 170,  22,  12,   2, 171,  93,   5,
          8, 157,  19,  60,  43,  35,  94,  80,  11,   2,  85, 200, 266, 105,
          2, 129, 198,  78])
tensor([[  0,   0,   0,  ...,  96, 265, 266],
    

In [32]:
# We don't sequencial model because LSTM returns more then one element , and sequencials all the layers should only return one element


class MyLSTM(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100,150,batch_first=True)
        self.fc = nn.Linear(150,vocab_size)
        
    def forward(self, x):
        embedded = self.embedding(x)
        internal_hidden_states,(final_hidden_state, final_cell_state) = self.lstm(embedded)
        output = self.fc(final_hidden_state.squeeze(0))
        return output


In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [34]:
model = MyLSTM(len(vocab))
model.to(device) 

MyLSTM(
  (embedding): Embedding(281, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=281, bias=True)
)

In [35]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [36]:
# training loop
for epoch in range(epochs):
    total_loss = 0 

    for batch_x , batch_y in dataloader:
        batch_x , batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        output = model(batch_x)

        loss = criterion(output , batch_y)

        loss.backward()

        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch: {epoch + 1 }: {total_loss:.4f}")

Epoch: 1: 165.5235
Epoch: 2: 145.2410
Epoch: 3: 132.1013
Epoch: 4: 120.1817
Epoch: 5: 108.5615
Epoch: 6: 96.6675
Epoch: 7: 86.7035
Epoch: 8: 77.2577
Epoch: 9: 68.3797
Epoch: 10: 60.6719
Epoch: 11: 53.5141
Epoch: 12: 46.4959
Epoch: 13: 41.0642
Epoch: 14: 36.3711
Epoch: 15: 31.6648
Epoch: 16: 28.4485
Epoch: 17: 24.6509
Epoch: 18: 22.2283
Epoch: 19: 19.3152
Epoch: 20: 17.6871
Epoch: 21: 15.9676
Epoch: 22: 14.0962
Epoch: 23: 13.0308
Epoch: 24: 11.8913
Epoch: 25: 10.8709
Epoch: 26: 10.0721
Epoch: 27: 9.4996
Epoch: 28: 8.8300
Epoch: 29: 8.2775
Epoch: 30: 7.8355
Epoch: 31: 7.6549
Epoch: 32: 7.2823
Epoch: 33: 6.8475
Epoch: 34: 6.4867
Epoch: 35: 6.2802
Epoch: 36: 6.2542
Epoch: 37: 5.9111
Epoch: 38: 5.6285
Epoch: 39: 5.8008
Epoch: 40: 5.4518
Epoch: 41: 5.3725
Epoch: 42: 5.1450
Epoch: 43: 5.1003
Epoch: 44: 5.5364
Epoch: 45: 4.9072
Epoch: 46: 5.0124
Epoch: 47: 4.8340
Epoch: 48: 4.5472
Epoch: 49: 4.4310
Epoch: 50: 4.3167


In [49]:
# Prediction 

def prediction(model,vocab,text):
    # tokenize 
    tokenized_text = tokenizer(text.lower())

    # text -> numerical indices
    numerical_text = text_to_indices(tokenized_text,vocab)
    
    # padding
    padded_text = torch.tensor([0]*(max_length-len(numerical_text)) + numerical_text,dtype=torch.long).unsqueeze(0)

    padded_text = padded_text.to(device)
    # send to model 
    output = model(padded_text)

    # predict index 
    value , index = torch.max(output,dim=1)

    # merge with text 
    return text + " " + list(vocab.keys())[index]


In [ ]:
prediction(model, vocab, "Will Deep Learning")

62


'The course follows a monthly subscription'

In [61]:
import time

num_tokens = 3
input_text = "ou can mail"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)

ou can mail us
ou can mail us at
ou can mail us at nitish.campusx@gmail.com


In [54]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [57]:
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")

Model Accuracy: 95.61%
